In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
proc_dir = root / "data" / "processed" / "swat"
stream_dir = root / "data" / "stream" / "swat"
stream_dir.mkdir(parents=True, exist_ok=True)

# 这里用 transaction 对齐后的标签长度来确定总窗口数
y = pd.read_csv(proc_dir / "y_filled.csv").iloc[:, 0]

lag = 2
tx_labels = y.iloc[lag:].reset_index(drop=True)

window_size = 2000
step_size = 1000

windows = []
for start in range(0, len(tx_labels) - window_size + 1, step_size):
    end = start + window_size
    yw = tx_labels.iloc[start:end]

    windows.append({
        "window_id": len(windows),
        "start_idx": start,
        "end_idx": end,
        "n_samples": len(yw),
        "n_attack": int((yw == 1).sum()),
        "n_normal": int((yw == 0).sum()),
        "attack_ratio": float((yw == 1).mean())
    })

windows_df = pd.DataFrame(windows)

print("窗口总数:", len(windows_df))
print(windows_df.head(10))
print(windows_df["attack_ratio"].describe())

windows_df.to_csv(stream_dir / "full_stream_windows_2000_1000.csv", index=False)
print("saved:", stream_dir / "full_stream_windows_2000_1000.csv")

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"

windows_df = pd.read_csv(stream_dir / "full_stream_windows_2000_1000.csv")

normal_df = windows_df[windows_df["attack_ratio"] == 0].copy()
mixed_df = windows_df[(windows_df["attack_ratio"] > 0) & (windows_df["attack_ratio"] < 1)].copy()
attack_df = windows_df[windows_df["attack_ratio"] == 1].copy()

print("纯正常窗口数:", len(normal_df))
print("混合窗口数:", len(mixed_df))
print("纯攻击窗口数:", len(attack_df))

print("\nnormal head:")
print(normal_df.head(5))

print("\nmixed head:")
print(mixed_df.head(5))

print("\nattack head:")
print(attack_df.head(5))

normal_df.to_csv(stream_dir / "windows_normal.csv", index=False)
mixed_df.to_csv(stream_dir / "windows_mixed.csv", index=False)
attack_df.to_csv(stream_dir / "windows_attack.csv", index=False)

print("\nsaved:", stream_dir / "windows_normal.csv")
print("saved:", stream_dir / "windows_mixed.csv")
print("saved:", stream_dir / "windows_attack.csv")

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"

normal_df = pd.read_csv(stream_dir / "windows_normal.csv")
mixed_df = pd.read_csv(stream_dir / "windows_mixed.csv")
attack_df = pd.read_csv(stream_dir / "windows_attack.csv")

# 1) 纯正常：前5个
normal_sel = normal_df.head(5).copy()
normal_sel["window_type"] = "normal"

# 2) 混合：围绕 0.2 / 0.5 / 0.8 各挑2个最近的
targets = [0.2, 0.5, 0.8]
mixed_parts = []
for t in targets:
    tmp = mixed_df.copy()
    tmp["dist"] = (tmp["attack_ratio"] - t).abs()
    tmp = tmp.sort_values("dist").head(2).copy()
    tmp["target_attack_ratio"] = t
    mixed_parts.append(tmp)

mixed_sel = pd.concat(mixed_parts, axis=0).drop_duplicates(subset=["window_id"]).copy()
mixed_sel["window_type"] = "mixed"

# 3) 纯攻击：前5个
attack_sel = attack_df.head(5).copy()
attack_sel["window_type"] = "attack"

exp_windows = pd.concat([normal_sel, mixed_sel, attack_sel], axis=0, ignore_index=True)

print("实验窗口数量:", len(exp_windows))
print(exp_windows[["window_id", "window_type", "start_idx", "end_idx", "attack_ratio"]])

exp_windows.to_csv(stream_dir / "exp_windows_round1.csv", index=False)
print("saved:", stream_dir / "exp_windows_round1.csv")

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
proc_dir = root / "data" / "processed" / "swat"
stream_dir = root / "data" / "stream" / "swat"
window_data_dir = stream_dir / "window_binary_data"
window_data_dir.mkdir(parents=True, exist_ok=True)

df_binary = pd.read_csv(proc_dir / "X_filled_binary.csv")
exp_windows = pd.read_csv(stream_dir / "exp_windows_round1.csv")

lag = 2

for _, row in exp_windows.iterrows():
    wid = int(row["window_id"])
    start = int(row["start_idx"])
    end = int(row["end_idx"])
    
    # 为了后面 build_transactions(lag=2)，这里多取前面 lag 行
    local_df = df_binary.iloc[start : end + lag].reset_index(drop=True)
    
    save_path = window_data_dir / f"window_{wid}_binary.csv"
    local_df.to_csv(save_path, index=False)

print("保存窗口数:", len(exp_windows))
print("示例文件:")
for p in sorted(window_data_dir.glob("window_*_binary.csv"))[:5]:
    print(p.name)

In [ ]:
import sys
import pickle
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

from src.data.transaction_utils import build_transactions

stream_dir = root / "data" / "stream" / "swat"
window_data_dir = stream_dir / "window_binary_data"
window_tx_dir = stream_dir / "window_transactions"
window_tx_dir.mkdir(parents=True, exist_ok=True)

# 先选一个 normal 和一个 mixed
test_window_ids = [0, 998]

for wid in test_window_ids:
    df_local = pd.read_csv(window_data_dir / f"window_{wid}_binary.csv")
    tx = build_transactions(df_local, lag=2)

    print(f"window {wid} -> transactions 数量:", len(tx))
    print(f"window {wid} -> 第1条前20项:", tx[0][:20])

    with open(window_tx_dir / f"window_{wid}_transactions.pkl", "wb") as f:
        pickle.dump(tx, f)

    print("saved:", window_tx_dir / f"window_{wid}_transactions.pkl")
    print("-" * 50)

In [ ]:
import pickle
import pandas as pd
from pathlib import Path
from mlxtend.preprocessing import TransactionEncoder

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_tx_dir = stream_dir / "window_transactions"
window_onehot_dir = stream_dir / "window_onehot"
window_onehot_dir.mkdir(parents=True, exist_ok=True)

test_window_ids = [0, 998]

for wid in test_window_ids:
    with open(window_tx_dir / f"window_{wid}_transactions.pkl", "rb") as f:
        tx = pickle.load(f)

    te = TransactionEncoder()
    arr = te.fit(tx).transform(tx)
    df_onehot = pd.DataFrame(arr, columns=te.columns_)

    print(f"window {wid} -> one-hot shape:", df_onehot.shape)
    print(f"window {wid} -> 前10列:", df_onehot.columns[:10].tolist())

    save_path = window_onehot_dir / f"window_{wid}_onehot.csv"
    df_onehot.to_csv(save_path, index=False)
    print("saved:", save_path)
    print("-" * 50)

In [ ]:
import pandas as pd
from pathlib import Path
from mlxtend.frequent_patterns import fpgrowth

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_onehot_dir = stream_dir / "window_onehot"
window_freq_dir = stream_dir / "window_freq_items"
window_freq_dir.mkdir(parents=True, exist_ok=True)

test_window_ids = [0, 998]

for wid in test_window_ids:
    df_onehot = pd.read_csv(window_onehot_dir / f"window_{wid}_onehot.csv")

    freq_items = fpgrowth(
        df_onehot,
        min_support=0.5,
        use_colnames=True,
        max_len=2
    )

    freq_items = freq_items.sort_values("support", ascending=False).reset_index(drop=True)

    print(f"window {wid} -> 频繁项集数量:", len(freq_items))
    print(freq_items.head(10))

    save_path = window_freq_dir / f"window_{wid}_freq_items.csv"
    freq_items.to_csv(save_path, index=False)
    print("saved:", save_path)
    print("-" * 60)

In [ ]:
import pickle
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
proc_dir = root / "data" / "processed" / "swat"
stream_dir = root / "data" / "stream" / "swat"
window_tx_dir = stream_dir / "window_transactions"
window_labeled_dir = stream_dir / "window_labeled_transactions"
window_labeled_dir.mkdir(parents=True, exist_ok=True)

wid = 998

# 读取窗口表
exp_windows = pd.read_csv(stream_dir / "exp_windows_round1.csv")
row = exp_windows[exp_windows["window_id"] == wid].iloc[0]
start = int(row["start_idx"])
end = int(row["end_idx"])

# 读取 transaction 对齐标签
y = pd.read_csv(proc_dir / "y_filled.csv").iloc[:, 0]
lag = 2
tx_labels = y.iloc[lag:].reset_index(drop=True)

labels_window = tx_labels.iloc[start:end].reset_index(drop=True)

# 读取该窗口 transactions
with open(window_tx_dir / f"window_{wid}_transactions.pkl", "rb") as f:
    tx = pickle.load(f)

# 加标签项
tx_labeled = []
for one_tx, label in zip(tx, labels_window):
    new_tx = list(one_tx)
    new_tx.append("LABEL_ATTACK" if label == 1 else "LABEL_NORMAL")
    tx_labeled.append(new_tx)

print("window 998 -> labeled transactions 数量:", len(tx_labeled))
print("window 998 -> 标签分布:")
print(labels_window.value_counts())
print("window 998 -> 第1条最后5项:", tx_labeled[0][-5:])

with open(window_labeled_dir / f"window_{wid}_transactions_labeled.pkl", "wb") as f:
    pickle.dump(tx_labeled, f)

print("saved:", window_labeled_dir / f"window_{wid}_transactions_labeled.pkl")

In [ ]:
import pickle
import pandas as pd
from pathlib import Path
from mlxtend.preprocessing import TransactionEncoder

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_labeled_dir = stream_dir / "window_labeled_transactions"
window_labeled_onehot_dir = stream_dir / "window_labeled_onehot"
window_labeled_onehot_dir.mkdir(parents=True, exist_ok=True)

wid = 998

with open(window_labeled_dir / f"window_{wid}_transactions_labeled.pkl", "rb") as f:
    tx_labeled = pickle.load(f)

te = TransactionEncoder()
arr = te.fit(tx_labeled).transform(tx_labeled)
df_labeled_onehot = pd.DataFrame(arr, columns=te.columns_)

print("window 998 -> labeled one-hot shape:", df_labeled_onehot.shape)
print("window 998 -> 最后10列:", df_labeled_onehot.columns[-10:].tolist())

save_path = window_labeled_onehot_dir / f"window_{wid}_labeled_onehot.csv"
df_labeled_onehot.to_csv(save_path, index=False)

print("saved:", save_path)

In [ ]:
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_labeled_onehot_dir = stream_dir / "window_labeled_onehot"

wid = 998
df_labeled_onehot = pd.read_csv(window_labeled_onehot_dir / f"window_{wid}_labeled_onehot.csv")

print("LABEL_ATTACK exists:", "LABEL_ATTACK" in df_labeled_onehot.columns)
print("LABEL_NORMAL exists:", "LABEL_NORMAL" in df_labeled_onehot.columns)

print("LABEL_ATTACK sum:", int(df_labeled_onehot["LABEL_ATTACK"].sum()))
print("LABEL_NORMAL sum:", int(df_labeled_onehot["LABEL_NORMAL"].sum()))

In [ ]:
import pandas as pd
from pathlib import Path
from mlxtend.frequent_patterns import fpgrowth, association_rules

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_labeled_onehot_dir = stream_dir / "window_labeled_onehot"
window_rule_dir = stream_dir / "window_label_rules"
window_rule_dir.mkdir(parents=True, exist_ok=True)

wid = 998
df_labeled_onehot = pd.read_csv(window_labeled_onehot_dir / f"window_{wid}_labeled_onehot.csv")

freq_items_lbl = fpgrowth(
    df_labeled_onehot,
    min_support=0.1,
    use_colnames=True,
    max_len=2
)

rules_lbl = association_rules(
    freq_items_lbl,
    metric="confidence",
    min_threshold=0.6
)

rules_attack = rules_lbl[
    (rules_lbl["antecedents"].apply(len) == 1) &
    (rules_lbl["consequents"].apply(lambda x: x == frozenset({"LABEL_ATTACK"})))
].copy()

rules_normal = rules_lbl[
    (rules_lbl["antecedents"].apply(len) == 1) &
    (rules_lbl["consequents"].apply(lambda x: x == frozenset({"LABEL_NORMAL"})))
].copy()

rules_attack["antecedent_str"] = rules_attack["antecedents"].apply(lambda x: list(x)[0])
rules_normal["antecedent_str"] = rules_normal["antecedents"].apply(lambda x: list(x)[0])

rules_attack = rules_attack.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

rules_normal = rules_normal.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

print("window 998 -> attack rules:", len(rules_attack))
print(rules_attack[["antecedent_str", "support", "confidence", "lift"]].head(10))

print("\nwindow 998 -> normal rules:", len(rules_normal))
print(rules_normal[["antecedent_str", "support", "confidence", "lift"]].head(10))

rules_attack.to_csv(window_rule_dir / f"window_{wid}_rules_attack.csv", index=False)
rules_normal.to_csv(window_rule_dir / f"window_{wid}_rules_normal.csv", index=False)

print("\nsaved:", window_rule_dir / f"window_{wid}_rules_attack.csv")
print("saved:", window_rule_dir / f"window_{wid}_rules_normal.csv")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_rule_dir = stream_dir / "window_label_rules"
window_pool_dir = stream_dir / "window_rule_pools"
window_pool_dir.mkdir(parents=True, exist_ok=True)

wid = 998

rules_attack = pd.read_csv(window_rule_dir / f"window_{wid}_rules_attack.csv")
rules_normal = pd.read_csv(window_rule_dir / f"window_{wid}_rules_normal.csv")

attack_top = rules_attack.head(10).copy()
normal_top = rules_normal.head(10).copy()

def confidence_to_weight(conf, eps=1e-6):
    conf = np.clip(conf, eps, 1 - eps)
    return float(np.log(conf / (1 - conf)))

def clipped_weight(conf, wmax=3.0):
    w = confidence_to_weight(conf)
    return float(np.clip(w, 0.0, wmax))

attack_top["consequent_str"] = "LABEL_ATTACK"
attack_top["formula"] = attack_top["antecedent_str"] + " => LABEL_ATTACK"
attack_top["weight"] = attack_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
attack_top["target_label"] = 1

normal_top["consequent_str"] = "LABEL_NORMAL"
normal_top["formula"] = normal_top["antecedent_str"] + " => LABEL_NORMAL"
normal_top["weight"] = normal_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
normal_top["target_label"] = 0

mixed_rule_pool = pd.concat([
    attack_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
    normal_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
], axis=0, ignore_index=True)

print("window 998 mixed_rule_pool 数量:", len(mixed_rule_pool))
print("weight describe:")
print(mixed_rule_pool["weight"].describe())
print(mixed_rule_pool[["formula", "confidence", "weight", "target_label"]].head(10))

save_path = window_pool_dir / f"window_{wid}_mixed_rule_pool.csv"
mixed_rule_pool.to_csv(save_path, index=False)
print("saved:", save_path)

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import pandas as pd

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.state_utils as state_utils
importlib.reload(state_utils)

from src.rl.state_utils import build_initial_rule_state

stream_dir = root / "data" / "stream" / "swat"
window_pool_dir = stream_dir / "window_rule_pools"
window_state_dir = stream_dir / "window_rule_states"
window_state_dir.mkdir(parents=True, exist_ok=True)

wid = 998
mixed_rule_pool = pd.read_csv(window_pool_dir / f"window_{wid}_mixed_rule_pool.csv")
window_state = build_initial_rule_state(mixed_rule_pool)

print("window 998 -> num_rules:", window_state["num_rules"])
print("window 998 -> weights:", window_state["weights"])
print("window 998 -> target_labels:", window_state["target_labels"])

save_path = window_state_dir / f"window_{wid}_rule_state.pkl"
with open(save_path, "wb") as f:
    pickle.dump(window_state, f)

print("saved:", save_path)

In [ ]:
import sys
import pickle
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

from src.data.transaction_utils import build_transactions

proc_dir = root / "data" / "processed" / "swat"
stream_dir = root / "data" / "stream" / "swat"
window_data_dir = stream_dir / "window_binary_data"
window_tx_dir = stream_dir / "window_transactions"
window_tx_dir.mkdir(parents=True, exist_ok=True)

wid = 1436

# 1) 如果还没保存这个窗口的二值数据，就先切出来
df_binary = pd.read_csv(proc_dir / "X_filled_binary.csv")
exp_windows = pd.read_csv(stream_dir / "exp_windows_round1.csv")

row = exp_windows[exp_windows["window_id"] == wid].iloc[0]
start = int(row["start_idx"])
end = int(row["end_idx"])
lag = 2

local_df = df_binary.iloc[start:end + lag].reset_index(drop=True)
local_df.to_csv(window_data_dir / f"window_{wid}_binary.csv", index=False)

# 2) 构造 transactions
tx = build_transactions(local_df, lag=2)

print(f"window {wid} attack_ratio:", row['attack_ratio'])
print(f"window {wid} -> transactions 数量:", len(tx))
print(f"window {wid} -> 第1条前20项:", tx[0][:20])

with open(window_tx_dir / f"window_{wid}_transactions.pkl", "wb") as f:
    pickle.dump(tx, f)

print("saved:", window_data_dir / f"window_{wid}_binary.csv")
print("saved:", window_tx_dir / f"window_{wid}_transactions.pkl")

In [ ]:
import pickle
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
proc_dir = root / "data" / "processed" / "swat"
stream_dir = root / "data" / "stream" / "swat"
window_tx_dir = stream_dir / "window_transactions"
window_labeled_dir = stream_dir / "window_labeled_transactions"
window_labeled_dir.mkdir(parents=True, exist_ok=True)

wid = 1436

# 读取窗口位置
exp_windows = pd.read_csv(stream_dir / "exp_windows_round1.csv")
row = exp_windows[exp_windows["window_id"] == wid].iloc[0]
start = int(row["start_idx"])
end = int(row["end_idx"])

# 对齐 transaction 标签
y = pd.read_csv(proc_dir / "y_filled.csv").iloc[:, 0]
lag = 2
tx_labels = y.iloc[lag:].reset_index(drop=True)
labels_window = tx_labels.iloc[start:end].reset_index(drop=True)

# 读取 transactions
with open(window_tx_dir / f"window_{wid}_transactions.pkl", "rb") as f:
    tx = pickle.load(f)

# 加标签项
tx_labeled = []
for one_tx, label in zip(tx, labels_window):
    new_tx = list(one_tx)
    new_tx.append("LABEL_ATTACK" if label == 1 else "LABEL_NORMAL")
    tx_labeled.append(new_tx)

print(f"window {wid} -> labeled transactions 数量:", len(tx_labeled))
print(f"window {wid} -> 标签分布:")
print(labels_window.value_counts())
print(f"window {wid} -> 第1条最后5项:", tx_labeled[0][-5:])

save_path = window_labeled_dir / f"window_{wid}_transactions_labeled.pkl"
with open(save_path, "wb") as f:
    pickle.dump(tx_labeled, f)

print("saved:", save_path)

In [ ]:
import pickle
import pandas as pd
from pathlib import Path
from mlxtend.preprocessing import TransactionEncoder

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_labeled_dir = stream_dir / "window_labeled_transactions"
window_labeled_onehot_dir = stream_dir / "window_labeled_onehot"
window_labeled_onehot_dir.mkdir(parents=True, exist_ok=True)

wid = 1436

with open(window_labeled_dir / f"window_{wid}_transactions_labeled.pkl", "rb") as f:
    tx_labeled = pickle.load(f)

te = TransactionEncoder()
arr = te.fit(tx_labeled).transform(tx_labeled)
df_labeled_onehot = pd.DataFrame(arr, columns=te.columns_)

print(f"window {wid} -> labeled one-hot shape:", df_labeled_onehot.shape)
print(f"window {wid} -> LABEL_ATTACK exists:", "LABEL_ATTACK" in df_labeled_onehot.columns)
print(f"window {wid} -> LABEL_NORMAL exists:", "LABEL_NORMAL" in df_labeled_onehot.columns)
print(f"window {wid} -> LABEL_ATTACK sum:", int(df_labeled_onehot["LABEL_ATTACK"].sum()))
print(f"window {wid} -> LABEL_NORMAL sum:", int(df_labeled_onehot["LABEL_NORMAL"].sum()))

save_path = window_labeled_onehot_dir / f"window_{wid}_labeled_onehot.csv"
df_labeled_onehot.to_csv(save_path, index=False)

print("saved:", save_path)

In [ ]:
import pandas as pd
from pathlib import Path
from mlxtend.frequent_patterns import fpgrowth, association_rules

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_labeled_onehot_dir = stream_dir / "window_labeled_onehot"
window_rule_dir = stream_dir / "window_label_rules"
window_rule_dir.mkdir(parents=True, exist_ok=True)

wid = 1436
df_labeled_onehot = pd.read_csv(window_labeled_onehot_dir / f"window_{wid}_labeled_onehot.csv")

freq_items_lbl = fpgrowth(
    df_labeled_onehot,
    min_support=0.1,
    use_colnames=True,
    max_len=2
)

rules_lbl = association_rules(
    freq_items_lbl,
    metric="confidence",
    min_threshold=0.6
)

rules_attack = rules_lbl[
    (rules_lbl["antecedents"].apply(len) == 1) &
    (rules_lbl["consequents"].apply(lambda x: x == frozenset({"LABEL_ATTACK"})))
].copy()

rules_normal = rules_lbl[
    (rules_lbl["antecedents"].apply(len) == 1) &
    (rules_lbl["consequents"].apply(lambda x: x == frozenset({"LABEL_NORMAL"})))
].copy()

rules_attack["antecedent_str"] = rules_attack["antecedents"].apply(lambda x: list(x)[0])
rules_normal["antecedent_str"] = rules_normal["antecedents"].apply(lambda x: list(x)[0])

rules_attack = rules_attack.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

rules_normal = rules_normal.sort_values(
    ["confidence", "lift", "support"],
    ascending=False
).reset_index(drop=True)

print(f"window {wid} -> attack rules:", len(rules_attack))
print(rules_attack[["antecedent_str", "support", "confidence", "lift"]].head(10))

print(f"\nwindow {wid} -> normal rules:", len(rules_normal))
print(rules_normal[["antecedent_str", "support", "confidence", "lift"]].head(10))

rules_attack.to_csv(window_rule_dir / f"window_{wid}_rules_attack.csv", index=False)
rules_normal.to_csv(window_rule_dir / f"window_{wid}_rules_normal.csv", index=False)

print("\nsaved:", window_rule_dir / f"window_{wid}_rules_attack.csv")
print("saved:", window_rule_dir / f"window_{wid}_rules_normal.csv")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
stream_dir = root / "data" / "stream" / "swat"
window_rule_dir = stream_dir / "window_label_rules"
window_pool_dir = stream_dir / "window_rule_pools"
window_pool_dir.mkdir(parents=True, exist_ok=True)

wid = 1436

rules_attack = pd.read_csv(window_rule_dir / f"window_{wid}_rules_attack.csv")
rules_normal = pd.read_csv(window_rule_dir / f"window_{wid}_rules_normal.csv")

attack_top = rules_attack.head(10).copy()
normal_top = rules_normal.head(9).copy()   # 这里正常规则只有9条

def confidence_to_weight(conf, eps=1e-6):
    conf = np.clip(conf, eps, 1 - eps)
    return float(np.log(conf / (1 - conf)))

def clipped_weight(conf, wmax=3.0):
    w = confidence_to_weight(conf)
    return float(np.clip(w, 0.0, wmax))

attack_top["consequent_str"] = "LABEL_ATTACK"
attack_top["formula"] = attack_top["antecedent_str"] + " => LABEL_ATTACK"
attack_top["weight"] = attack_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
attack_top["target_label"] = 1

normal_top["consequent_str"] = "LABEL_NORMAL"
normal_top["formula"] = normal_top["antecedent_str"] + " => LABEL_NORMAL"
normal_top["weight"] = normal_top["confidence"].apply(lambda x: clipped_weight(x, wmax=3.0))
normal_top["target_label"] = 0

mixed_rule_pool = pd.concat([
    attack_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
    normal_top[["antecedent_str", "consequent_str", "formula", "support", "confidence", "lift", "weight", "target_label"]],
], axis=0, ignore_index=True)

print(f"window {wid} mixed_rule_pool 数量:", len(mixed_rule_pool))
print("标签分布:")
print(mixed_rule_pool["target_label"].value_counts())
print(mixed_rule_pool[["formula", "confidence", "weight", "target_label"]].head(10))

save_path = window_pool_dir / f"window_{wid}_mixed_rule_pool.csv"
mixed_rule_pool.to_csv(save_path, index=False)
print("saved:", save_path)

In [ ]:
import sys
import importlib
import pickle
from pathlib import Path
import pandas as pd

root = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
sys.path.append(str(root))

import src.rl.state_utils as state_utils
importlib.reload(state_utils)

from src.rl.state_utils import build_initial_rule_state

stream_dir = root / "data" / "stream" / "swat"
window_pool_dir = stream_dir / "window_rule_pools"
window_state_dir = stream_dir / "window_rule_states"
window_state_dir.mkdir(parents=True, exist_ok=True)

wid = 1436
mixed_rule_pool = pd.read_csv(window_pool_dir / f"window_{wid}_mixed_rule_pool.csv")

window_state = build_initial_rule_state(mixed_rule_pool)

print(f"window {wid} -> num_rules:", window_state["num_rules"])
print(f"window {wid} -> weights:", window_state["weights"])
print(f"window {wid} -> target_labels:", window_state["target_labels"])

save_path = window_state_dir / f"window_{wid}_rule_state.pkl"
with open(save_path, "wb") as f:
    pickle.dump(window_state, f)

print("saved:", save_path)